# Kafka za streaming slika
Koristi zookeeper, kafka samo prima/ šalje ID-eve slika iz bucketa zbog optimizacije performansi

In [5]:
%%writefile ../kafka.docker-compose.yml
version: '3'

networks:
  cv-network:
    #name: "app-network"
    #driver: bridge  #TODO: remove this line after merging with docker-compose.yml
    external: true

volumes:
  kafka-data:
  kafka-secrets:
  zookeeper-data:
  zookeeper-log:
  zookeeper-secrets:

services:
  # --- Infrastructure Services (largely unchanged) ---

  
  zookeeper:
    image:  confluentinc/cp-zookeeper:latest
    container_name: zookeeper
    stop_grace_period: 30s 
    networks:
      - cv-network
    env_file:
        - ./Kafka/zookeeper.env
      
    volumes:
      - zookeeper-data:/var/lib/zookeeper/data
      - zookeeper-log:/var/lib/zookeeper/log
      - zookeeper-secrets:/etc/zookeeper/secrets


  kafka:
    image: confluentinc/cp-kafka:latest
    container_name: kafka
    networks:
      - cv-network
    depends_on:
      - zookeeper
    ports:
      - "9092:9092"
    env_file:
        - ./Kafka/kafka.env
      
    volumes:
      - kafka-data:/var/lib/kafka/data
      - kafka-secrets:/etc/kafka/secrets

Overwriting ../kafka.docker-compose.yml


# Env datoteke za konfiguriranje kontenjera

In [7]:
%%writefile ./kafka.env      
KAFKA_BROKER_ID=1
KAFKA_ZOOKEEPER_CONNECT=zookeeper:2181
KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka:29092,PLAINTEXT_HOST://10.124.17.151:9092
KAFKA_LISTENERS=PLAINTEXT://0.0.0.0:29092,PLAINTEXT_HOST://0.0.0.0:9092
KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=PLAINTEXT:PLAINTEXT,PLAINTEXT_HOST:PLAINTEXT
KAFKA_INTER_BROKER_LISTENER_NAME=PLAINTEXT
KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1

Overwriting ./kafka.env


In [4]:
%%writefile ./zookeeper.env
ZOOKEEPER_CLIENT_PORT=2181
ZOOKEEPER_TICK_TIME=2000

Overwriting ./zookeeper.env


# Modifikacija .env datoteke i dodavanje novog compose file-a na kraj

In [2]:
import os

def update_compose_file_env(env_file_path, compose_file_name):
    """
    Updates the .env file to include the specified compose file in the COMPOSE_FILE variable.

    Args:
        env_file_path (str): The path to the .env file.
        compose_file_name (str): The name of the compose file to add.
    """
    try:
        with open(env_file_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"Error: .env file not found at {env_file_path}")
        return

    updated = False
    with open(env_file_path, 'w') as f:
        for line in lines:
            if line.startswith("COMPOSE_FILE="):
                if compose_file_name not in line:
                    line = line.strip()
                    if line.endswith("="):
                        line += compose_file_name + "\n"
                    else:
                        line += ":" + compose_file_name + "\n"
                    updated = True
            f.write(line)

    if updated:
        print(f"Updated COMPOSE_FILE in {env_file_path}")
    else:
        print(f"COMPOSE_FILE already contains {compose_file_name} in {env_file_path}")

if __name__ == "__main__":
    env_file = "../.env"
    compose_file = "kafka.docker-compose.yml"
    update_compose_file_env(env_file, compose_file)

COMPOSE_FILE already contains kafka.docker-compose.yml in ../.env
